# Native kidney exchange optimization
Keep the original three donors, three patients, nine assignment bits, five QAOA layers, penalty factor two, 100-iteration limit and CVaR 0.7. Exact local CVaR and public Nelder-Mead replace the SDK session and optimizer; every vertex binds a fresh native program. Final samples are finite. Legal matching, raw compatibility and penalized cost remain separate.

In [ ]:
import json
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.optimization.kidney_exchange.kidney_exchange_problem.overview.model import (
    WEIGHTS,
)
from examples.synthesis.applications.optimization.kidney_exchange.kidney_exchange_problem.overview.plots import (
    plot_results,
)
from examples.synthesis.applications.optimization.kidney_exchange.kidney_exchange_problem.overview.quantum import (
    build_program,
)
from examples.synthesis.applications.optimization.kidney_exchange.kidney_exchange_problem.overview.workflow import (
    INITIAL_PARAMETERS,
    run_workflow,
)
from flagquantum.drawer import draw


In [ ]:
donors = ("donor1", "donor2", "donor3")
patients = ("patient1", "patient2", "patient3")
print(
    "Initial classical solver is commented out in the source: no classical solution asserted."
)
print("Compatibility matrix:")
for donor in range(3):
    print(donors[donor], WEIGHTS[3 * donor : 3 * donor + 3])
print("Constraints: each donor and patient participates at most once.")


In [ ]:
initial_program = build_program(INITIAL_PARAMETERS)
print("Five native layers, nine assignment bits; width objective.")
print("Synthesis-time real bindings; no reusable runtime-parameter claim.")


In [ ]:
print(initial_program.report["resources"])
print(draw(initial_program.circuit, show_all_qubits=True))


In [ ]:
workflow = run_workflow(maxiter=100, shots=1000, seed=2718)
print("Actual native evaluations:", len(workflow.trace))
print("Optimizer iterations:", workflow.optimizer.iterations)


In [ ]:
plot_paths = plot_results(workflow, Path("artifacts/development/kidney-exchange"))
display(Image(filename=str(plot_paths[0])))


In [ ]:
print("Top ten actual sampled observations:")
for row in workflow.samples[:10]:
    print(row)
assert sum(row.count for row in workflow.samples) == 1000


In [ ]:
display(Image(filename=str(plot_paths[1])))


In [ ]:
best = workflow.best_observed.assignment
print("Source score diagnostic (-penalized cost):", -best.penalized_cost)
print("Actual compatibility score:", best.score)
print("Conflict penalty:", 2 * best.conflict_pairs)
print("Feasible:", best.feasible)
print("Donor/patient table:")
for donor_label, matching_row in zip(donors, best.rows, strict=True):
    print(donor_label, matching_row)
print("Best observed legal assignment:", workflow.best_observed_feasible)
print("Separate exhaustive reference, calculated after sampling:", workflow.reference)
display(Image(filename=str(plot_paths[2])))
report = {
    "width": 9,
    "layers": 5,
    "maxiter": 100,
    "iterations": workflow.optimizer.iterations,
    "evaluations": len(workflow.trace),
    "quantile": 0.7,
    "shots": sum(row.count for row in workflow.samples),
    "best_word": best.word,
    "best_cost": best.penalized_cost,
    "best_score": best.score,
    "best_conflicts": best.conflict_pairs,
    "best_feasible": best.feasible,
    "reference_score": workflow.reference.score,
    "trace": [row.cost for row in workflow.trace],
}
print("ARTIFACT_JSON=" + json.dumps(report))
